---
title: "Non-Renewable Energy Production in the United States"
author: "Kevin Havis"
format: 
  html:
    code-fold: true
    theme: cosmo
    embed-resources: true
    fig-dpi: 150
  pdf:
    documentclass: article
    fig-align : "center"
    geometry: margin=0.2in
jupyter: python3
---

While renewable energy sources are gaining traction, non-renewable energy production remains a significant component of the United States' energy landscape. This heavy reliance on fossil fuels creates an unbalanced landscape of energy production across the country, potential weak points of energy security.

The immediately available observation shows how much Texas dominates non-renewable energy production, particularly in oil and natural gas. Other states like Wyoming and West Virginia also contribute significantly, primarily through coal production. In contrast, states in the Northeast and West Coast have relatively low non-renewable energy production, reflecting their greater emphasis on renewable energy sources.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import re

In [ ]:
import unicodedata

def clean_column_names(df, strip_accents=True, case='lower', remove_special=True, replace_spaces='_'):
    """
    Clean DataFrame column names similar to pyjanitor's clean_columns.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame
    strip_accents : bool, default True
        Remove accents from characters
    case : {'lower', 'upper', None}, default 'lower'
        Convert column names to specified case
    remove_special : bool, default True
        Remove special characters (keeping alphanumeric, spaces, and underscores)
    replace_spaces : str, default '_'
        Character(s) to replace spaces with. Use None to keep spaces

    Returns
    -------
    pd.DataFrame
        DataFrame with cleaned column names
    """
    # Work with a copy of column names
    new_columns = df.columns.tolist()

    # Strip accents if requested
    if strip_accents:
        new_columns = [
            ''.join(
                c for c in unicodedata.normalize('NFD', col)
                if unicodedata.category(c) != 'Mn'
            )
            for col in new_columns
        ]

    # Convert to Series for string operations
    col_series = pd.Series(new_columns)

    # Remove special characters
    if remove_special:
        col_series = col_series.str.replace(r'[^\w\s]', '', regex=True)

    # Replace spaces with specified character
    if replace_spaces is not None:
        col_series = col_series.str.replace(r'\s+', replace_spaces, regex=True)

    # Apply case conversion
    if case == 'lower':
        col_series = col_series.str.lower()
    elif case == 'upper':
        col_series = col_series.str.upper()

    # Remove leading/trailing underscores
    col_series = col_series.str.strip(replace_spaces if replace_spaces else ' ')

    df.columns = col_series.tolist()
    return df

In [ ]:
# Load data
df = pd.read_csv(r"hw_7/state_energy_production.csv")
df.drop(columns=["Demography", "Supply & Distribution", "Production"], inplace=True)

# Clean column names
df = clean_column_names(df)
df['state_name'] = df['state_code'].map({v: k for k, v in state_to_code.items()})

In [ ]:
# Set up column for hover text in viz
df["hover_text"] = (
    df["state_name"]
    + "<br>"
    + "Total Energy Production (trillion BTUs): "
    + df["total_energy_trillion_btu_"].astype(str)
    + "<br>"
    + "Crude Oil (thousand barrels per day): "
    + df["crude_oil_thousand_barrels_per_day_"].astype(str)
    + "<br>"
    + "Natural Gas (million cubic feet): "
    + df["gross_domestic_product_$_billion_"].astype(str)
    + "<br>"
    + "Coal (short tons): "
    + df["coal_thousand_short_tons_"].astype(str)
)

In [ ]:
# Visualization
fig = go.Figure()

fig.add_trace(
    go.Choropleth(
        locations=df["state_code"],  # Uses the state code to get the graphic to render
        z=df["total_energy_trillion_btu_"],  # Intensity by federal funding
        locationmode="USA-states",
        text=df["hover_text"],
        showscale=True,
        hovertemplate="%{text}<extra></extra>",
    )
)

fig.update_layout(
    title=dict(
        text="United States Non-Renewable Energy Production by State",
    ),
    geo=dict(scope="usa"),
    width=800,  # Wider to accommodate multiple colorbars
    height=600,
)

fig.show()
